# 🍃 Nhận diện bệnh cây trồng — EfficientNetB3 + PlantVillage (87K ảnh, 38 lớp)

Notebook huấn luyện hoàn chỉnh cho Kaggle. Sau khi chạy xong sẽ xuất ra:
- `plant_disease_model.keras` — model dùng cho backend web app
- `class_mapping.json` — ánh xạ `category_id (int)` ↔ tên lớp (bắt buộc cho backend)
- `plant_disease_export.zip` — gói tất cả để tải về

**Cách dùng:** Add dataset `vipoooool/new-plant-diseases-dataset` vào notebook, bật **Accelerator = GPU T4**, bật **Internet** (Settings → Internet).


In [ ]:
import os, json, time
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB3
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
print('Num GPUs:', len(tf.config.list_physical_devices('GPU')))


In [ ]:
# ============================================================
# 1. Đường dẫn dữ liệu & ánh xạ class -> category_id
# ============================================================
BASE = '/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)'
TRAIN_DIR = os.path.join(BASE, 'train')
VALID_DIR = os.path.join(BASE, 'valid')
TEST_DIR  = os.path.join(BASE, 'test')

class_names = sorted(os.listdir(TRAIN_DIR))
class_to_id = {name: i for i, name in enumerate(class_names)}
id_to_class = {str(i): name for name, i in class_to_id.items()}

mapping = {'class_to_id': class_to_id, 'id_to_class': id_to_class}
with open('class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, indent=2, ensure_ascii=False)

print('Số lớp:', len(class_names))
print('Ví dụ:', list(class_names)[:5])
print('category_id của lớp đầu tiên:', class_to_id[class_names[0]])


In [ ]:
# ============================================================
# 2. Nạp dữ liệu + augmentation
# ============================================================
IMG_SIZE = 300   # kích thước gốc của EfficientNetB3
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    label_mode='int', shuffle=True, seed=42)
valid_ds = keras.utils.image_dataset_from_directory(
    VALID_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    label_mode='int', shuffle=False)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
    label_mode='int', shuffle=False)

def normalize(x, y):
    return tf.cast(x, tf.float32) / 255.0, y

train_ds = train_ds.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
valid_ds = valid_ds.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(normalize, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
])


In [ ]:
# ============================================================
# 3. Xây dựng model: EfficientNetB3 (transfer learning)
# ============================================================
base_model = EfficientNetB3(weights='imagenet', include_top=False,
                            input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()


In [ ]:
# ============================================================
# 4. Huấn luyện phần đầu (head) — backbone đóng băng
# ============================================================
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
    keras.callbacks.ModelCheckpoint('plant_disease_best.keras', monitor='val_accuracy', save_best_only=True),
]

EPOCHS_HEAD = 5
t0 = time.time()
history = model.fit(train_ds, validation_data=valid_ds,
                    epochs=EPOCHS_HEAD, callbacks=callbacks)
print(f'Thời gian huấn luyện head: {(time.time()-t0)/60:.1f} phút')


In [ ]:
# ============================================================
# 5. Fine-tune: mở khóa phần lớn backbone, learning rate nhỏ
# ============================================================
base_model.trainable = True
for layer in base_model.layers[:100]:   # giữ 100 layer đầu đóng băng
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

EPOCHS_FT = 8
t0 = time.time()
history_ft = model.fit(train_ds, validation_data=valid_ds,
                       epochs=EPOCHS_FT, callbacks=callbacks)
print(f'Thời gian fine-tune: {(time.time()-t0)/60:.1f} phút')


In [ ]:
# ============================================================
# 6. Đánh giá trên test set
# ============================================================
best = keras.models.load_model('plant_disease_best.keras')
loss, acc = best.evaluate(test_ds)
print(f'Test accuracy: {acc:.4f}  |  Test loss: {loss:.4f}')

from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for x, y in test_ds:
    p = np.argmax(best.predict(x, verbose=0), axis=1)
    y_true.extend(y.numpy())
    y_pred.extend(p)

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))


In [ ]:
# ============================================================
# 7. Demo dự đoán 1 ảnh (kiểm tra pipeline giống backend)
# ============================================================
from tensorflow.keras.preprocessing import image as kimage

sample_dir = os.path.join(TEST_DIR, class_names[0])
sample_path = os.path.join(sample_dir, os.listdir(sample_dir)[0])
img = kimage.load_img(sample_path, target_size=(IMG_SIZE, IMG_SIZE))
x = kimage.img_to_array(img) / 255.0
x = np.expand_dims(x, axis=0)

probs = best.predict(x, verbose=0)[0]
cid = int(np.argmax(probs))
print('Ảnh mẫu  :', os.path.basename(sample_path))
print('Dự đoán  :', id_to_class[str(cid)])
print('category_id:', cid)
print('confidence :', float(probs[cid]))
plt.imshow(kimage.img_to_array(img).astype('uint8'))
plt.title(f'{id_to_class[str(cid)]} ({probs[cid]:.3f})')
plt.axis('off')
plt.show()


In [ ]:
# ============================================================
# 8. Xuất model + mapping, đóng gói để tải về
# ============================================================
best.save('plant_disease_model.keras')          # dùng cho backend

# (Tùy chọn) xuất TFLite nếu sau này làm app mobile
converter = tf.lite.TFLiteConverter.from_keras_model(best)
tflite_model = converter.convert()
with open('plant_disease_model.tflite', 'wb') as f:
    f.write(tflite_model)

import zipfile
with zipfile.ZipFile('plant_disease_export.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('plant_disease_model.keras')
    z.write('plant_disease_model.tflite')
    z.write('class_mapping.json')

print('Xuất xong! Hãy tải về: plant_disease_export.zip (gồm model .keras, .tflite, class_mapping.json)')
